In [0]:
df = spark.table('predictstockprices.plstocks.silver_stocks_price')
display(df)

In [0]:
## 📈 Gold Table: stock_performance_summary
# Ta tabela agreguje kluczowe wskaźniki dotyczące wyników akcji:
#
# 🔹 ROI 30/90/365 dni -> Return on Investment 
#    ROI = (Close - Open) / Open
#
# 🔹 CAGR -> Compound Annual Growth Rate 
#    CAGR = (Ending Value / Beginning Value) ^ (1 / Number of Years) - 1
#
# 🔹 Max Drawdown -> Największy procentowy spadek od lokalnego szczytu do dołka
#
# 🔹 Best/Worst Day -> Najlepszy i najgorszy jednodniowy zwrot
#
# 🔹 Dividendy -> średnia dywidenda, stopa dywidendy, suma wypłat, streak lat
#
# 🔹 Wskaźniki finansowe -> średnie P/E, P/B, EV/EBITDA, EV/Sales


In [0]:
# ==============================
# GOLD Table: stock_performance_summary
# 
# What you can see in this table:
# - daily ROI and average ROI over 30/90/365 days
#   ROI = (Close - Open) / Open
# - CAGR (Compound Annual Growth Rate)
#   CAGR = (Ending Price / Starting Price)^(1/Years) - 1
# - Max drawdown: largest % decline from a peak to a trough
# - Dividend history: average per share, total paid, last payment date, years with dividend
# - Financial ratios: P/E, P/B, EV/EBITDA, EV/Sales
# - Sector info
# ==============================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -----------------------------
# 1. Load source tables
# -----------------------------
prices = spark.table("predictstockprices.plstocks.silver_stocks_price") \
              .withColumn("close", F.col("CLOSE").cast("double")) \
              .withColumn("open", F.col("OPEN").cast("double")) \
              .withColumn("date", F.col("DATE").cast("date"))

sector = spark.table("predictstockprices.plstocks.silver_sector_lookup") \
              .select("ticker", "sector").dropDuplicates(["ticker"])

dividends = spark.table("predictstockprices.plstocks.silver_dividend_history")

ratios = spark.table("predictstockprices.plstocks.silver_financial_ratios")

# -----------------------------
# 2. Calculate daily ROI and rolling averages
# -----------------------------
prices = prices.withColumn("daily_roi", (F.col("close") - F.col("open")) / F.col("open"))

window_30 = Window.partitionBy("TICKER").orderBy("date").rowsBetween(-29, 0)
window_90 = Window.partitionBy("TICKER").orderBy("date").rowsBetween(-89, 0)
window_365 = Window.partitionBy("TICKER").orderBy("date").rowsBetween(-364, 0)

prices = prices.withColumn("avg_roi_30d", F.avg("daily_roi").over(window_30)) \
               .withColumn("avg_roi_90d", F.avg("daily_roi").over(window_90)) \
               .withColumn("avg_roi_365d", F.avg("daily_roi").over(window_365))

# Aggregate ROI per ticker
df_roi = prices.groupBy("TICKER").agg(
    F.avg("avg_roi_30d").alias("avg_roi_30d"),
    F.avg("avg_roi_90d").alias("avg_roi_90d"),
    F.avg("avg_roi_365d").alias("avg_roi_365d")
)

# -----------------------------
# 3. CAGR
# -----------------------------
df_cagr = prices.groupBy("TICKER").agg(
    F.first("close").alias("start_price"),
    F.last("close").alias("end_price"),
    (F.year(F.max("date")) - F.year(F.min("date")) + 1).alias("years")
)
df_cagr = df_cagr.withColumn("cagr", (F.col("end_price") / F.col("start_price")) ** (1 / F.col("years")) - 1)

# -----------------------------
# 4. Max Drawdown
# -----------------------------
window_ticker = Window.partitionBy("TICKER").orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
prices = prices.withColumn("running_max", F.max("close").over(window_ticker))
prices = prices.withColumn("drawdown", (F.col("close") - F.col("running_max")) / F.col("running_max"))
df_max_drawdown = prices.groupBy("TICKER").agg(F.min("drawdown").alias("max_drawdown"))

# -----------------------------
# 5. Dividend aggregation
# -----------------------------
df_dividends = dividends.groupBy("ticker").agg(
    F.avg("dividend_per_share").alias("avg_dividend_per_share"),
    F.sum("dividend_value").alias("total_dividends_paid"),
    F.max("payment_date").alias("last_dividend_payment_date"),
    F.count("dividend_year").alias("dividend_years_count")
)

# -----------------------------
# 6. Financial ratios
# -----------------------------
df_ratios = ratios.groupBy("ticker").agg(
    F.avg("price_earnings").alias("avg_pe_ratio"),
    F.avg("price_book_value").alias("avg_pb_ratio"),
    F.avg("ev_ebitda").alias("avg_ev_ebitda"),
    F.avg("ev_sales").alias("avg_ev_sales")
)

# -----------------------------
# 7. Build GOLD table
# -----------------------------
gold_df = df_cagr.join(df_max_drawdown, "TICKER", "left") \
                  .join(df_roi, "TICKER", "left") \
                  .join(df_dividends, df_cagr.TICKER == df_dividends.ticker, "left") \
                  .join(df_ratios, df_cagr.TICKER == df_ratios.ticker, "left") \
                  .join(sector, df_cagr.TICKER == sector.ticker, "left") \
                  .select(
                      df_cagr.TICKER.alias("ticker"),
                      "sector",
                      "cagr",
                      "max_drawdown",
                      "avg_roi_30d",
                      "avg_roi_90d",
                      "avg_roi_365d",
                      "avg_dividend_per_share",
                      "total_dividends_paid",
                      "last_dividend_payment_date",
                      "dividend_years_count",
                      "avg_pe_ratio",
                      "avg_pb_ratio",
                      "avg_ev_ebitda",
                      "avg_ev_sales"
                  )

# -----------------------------
# 8. Save GOLD table
# -----------------------------
gold_df.write.format("delta").mode("overwrite").saveAsTable("predictstockprices.plstocks.gold_stock_performance_summary")


In [0]:
display(spark.table("predictstockprices.plstocks.gold_stock_performance_summary"))

In [0]:
#spark.sql("DROP TABLE IF EXISTS predictstockprices.plstocks.gold_stock_performance_summary")

In [0]:
display(spark.table("predictstockprices.plstocks.silver_sector_lookup"))  

In [0]:
from pyspark.sql import functions as F

df = spark.table("predictstockprices.plstocks.silver_stocks_price") \
    .filter(F.col("TICKER").startswith("KR"))

display(df)

In [0]:
display(spark.table("predictstockprices.plstocks.silver_dividend_history"))